In [ ]:
setwd("/content/buckets/b1/exp")
experimento_folder <- PARAM$experimento
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
Sys.time()
require( "data.table" )

# leo el dataset
dataset <- fread("~/buckets/b1/datasets/competencia_02_crudo.csv.gz" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
  "pos" = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 )
]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente
]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
  ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
  clase_ternaria := "BAJA+1"
]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
  & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
  clase_ternaria := "BAJA+2"
]

# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

rm(dsimple)
gc()
Sys.time()

In [ ]:
#Veo si en 202006 active_quarter son todos 0
dataset[foto_mes == 202006, .N, by = active_quarter]

In [ ]:
#Veo si en 202006 internet son todos 0
dataset[foto_mes == 202006, .N, by = internet]

# si el campo internet invirtió sus valores en 202010

In [ ]:
# 3) Matriz de transición 202009 → 202010 (más visual)

d9  <- dataset[foto_mes == 202009, .(numero_de_cliente, iternet_09 = iternet)]
d10 <- dataset[foto_mes == 202010, .(numero_de_cliente, iternet_10 = iternet)]

trans_0910 <- d9[d10, on = "numero_de_cliente", nomatch = 0L][
  iternet_09 %in% 0:1 & iternet_10 %in% 0:1,
  .N, by = .(iternet_09, iternet_10)
][order(iternet_09, iternet_10)]

trans_0910[, pct := N / sum(N)]
trans_0910

In [ ]:
# Si flip_rate ≈ 1, se invirtió para (casi) todos.
# Si hay same > 0 o out_of_domain > 0, hay excepciones o valores raros.

# ordenar para que el lag sea del mes anterior disponible del mismo cliente

setorder(dataset, numero_de_cliente, foto_mes)

# valor previo de iternet por cliente
dataset[, iternet_lag1 := shift(iternet, 1L, NA, "lag"), by = numero_de_cliente]

# evaluar sólo 202010 con lag disponible y binario
eval_202010 <- dataset[
  foto_mes == 202010 & !is.na(iternet) & !is.na(iternet_lag1),
  .(
    N = .N,
    flips = sum(iternet %in% 0:1 & iternet_lag1 %in% 0:1 & iternet == 1L - iternet_lag1),
    same  = sum(iternet %in% 0:1 & iternet_lag1 %in% 0:1 & iternet == iternet_lag1),
    out_of_domain = sum(!(iternet %in% 0:1) | !(iternet_lag1 %in% 0:1))
  )
][, `:=`(flip_rate = flips/N, same_rate = same/N)]
eval_202010

# si el campo tmobile_app invirtió sus valores en 202010

In [ ]:
# 3) Matriz de transición 202009 → 202010 (más visual)

d9  <- dataset[foto_mes == 202009, .(numero_de_cliente, tmobile_app_09 = tmobile_app)]
d10 <- dataset[foto_mes == 202010, .(numero_de_cliente, tmobile_app_10 = tmobile_app)]

trans_0910 <- d9[d10, on = "numero_de_cliente", nomatch = 0L][
  tmobile_app_09 %in% 0:1 & tmobile_app_10 %in% 0:1,
  .N, by = .(tmobile_app_09, tmobile_app_10)
][order(tmobile_app_09, tmobile_app_10)]

trans_0910[, pct := N / sum(N)]
trans_0910

In [ ]:
# Si flip_rate ≈ 1, se invirtió para (casi) todos.
# Si hay same > 0 o out_of_domain > 0, hay excepciones o valores raros.

# ordenar para que el lag sea del mes anterior disponible del mismo cliente

setorder(dataset, numero_de_cliente, foto_mes)

# valor previo de tmobile_app por cliente
dataset[, tmobile_app_lag1 := shift(tmobile_app, 1L, NA, "lag"), by = numero_de_cliente]

# evaluar sólo 202010 con lag disponible y binario
eval_202010 <- dataset[
  foto_mes == 202010 & !is.na(tmobile_app) & !is.na(tmobile_app_lag1),
  .(
    N = .N,
    flips = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == 1L - tmobile_app_lag1),
    same  = sum(tmobile_app %in% 0:1 & tmobile_app_lag1 %in% 0:1 & tmobile_app == tmobile_app_lag1),
    out_of_domain = sum(!(tmobile_app %in% 0:1) | !(tmobile_app_lag1 %in% 0:1))
  )
][, `:=`(flip_rate = flips/N, same_rate = same/N)]
eval_202010

# pandemia


Marcá como meses “críticos de pandemia” → 202003 a 202005

Marcá como meses “de recuperación atípica” → 202006 a 202012

Si hacés modelos o tendencia, podés:

excluirlos del entrenamiento,

o introducir una variable pandemia = 1 para esos meses,

o suavizar con medias móviles para evitar que esos saltos sesguen la tendencia.

In [ ]:
#Veo si en 201905 mrentabilidad son todos 0
dataset[foto_mes == 201905, .N, by = rentabilidad]

In [ ]:
#Veo si en 201910 mrentabilidad son todos 0
dataset[foto_mes == 201910, .N, by = rentabilidad]

In [ ]:
#Veo si en 202006 mrentabilidad son todos 0
dataset[foto_mes == 202006, .N, by = rentabilidad]

In [ ]:
#Veo si en 201905 mrentabilidad_annual son todos 0
dataset[foto_mes == 201905, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 201910 mrentabilidad_annual son todos 0
dataset[foto_mes == 201910, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 202006 mrentabilidad_annual son todos 0
dataset[foto_mes == 202006, .N, by = mrentabilidad_annual]

In [ ]:
#Veo si en 201905 mactivos_margen son todos 0
dataset[foto_mes == 201905, .N, by = mactivos_margen]

In [ ]:
#Veo si en 201910 mactivos_margen son todos 0
dataset[foto_mes == 201910, .N, by = mactivos_margen]

In [ ]:
#Veo si en 202006 mactivos_margen son todos 0
dataset[foto_mes == 202006, .N, by = mactivos_margen]

In [ ]:
#Veo si en 201905 mpasivos_margen son todos 0
dataset[foto_mes == 201905, .N, by = mpasivos_margen]

In [ ]:
#Veo si en 201910 mpasivos_margen son todos 0
dataset[foto_mes == 201910, .N, by = mpasivos_margen]

In [ ]:
#Veo si en 202006 mpasivos_margen son todos 0
dataset[foto_mes == 202006, .N, by = mpasivos_margen]

# detectar columnas todas en 0 para un periodo

In [ ]:
# Requiere data.table
library(data.table)

# dt = tu dataset (data.table) con al menos: foto_mes
# Excluí claves/no numéricas:
id_cols <- c("numero_de_cliente","foto_mes","clase_ternaria")
num_cols <- setdiff(names(dataset), id_cols)
num_cols <- num_cols[vapply(dataset[, ..num_cols], is.numeric, logical(1))]

# Conteo de no-cero por columna y mes
nz_counts <- dataset[ , lapply(.SD, function(v) sum(v != 0, na.rm = TRUE)),
                     by = .(foto_mes), .SDcols = num_cols]

# Total de filas por mes
n_by <- dataset[ , .N, by = .(foto_mes)]

# Paso a formato largo
library(data.table)
res <- melt(nz_counts, id.vars = "foto_mes",
            variable.name = "columna", value.name = "nonzero")
res <- res[n_by, on = "foto_mes"]
setnames(res, "N", "n_filas")

# Flag principal: todo cero (al menos una observación presente)
res[, all_zero := (nonzero == 0)]
res[]
